# 11 – Game-Theoretic Analysis

This notebook solves the attacker–defender games produced by `10_payoff.ipynb` separately for every payoff-weight and attacker-cost scenario. It supports `quick`, `pilot`, and `full` outputs and automatically selects the most complete available run.

For each scenario it computes:

- attacker and defender best responses;
- pure-strategy Nash equilibria;
- mixed-strategy equilibria through support enumeration when `nashpy` is installed;
- expected attacker and defender utilities;
- equilibrium support changes across sensitivity scenarios.

Quick-mode results contain only one attacker strategy, so the attacker's strategic choice is trivial. They validate the analysis pipeline but cannot support final equilibrium claims. For multi-strategy pilot/full analysis, install `nashpy` if necessary with `%pip install nashpy`, restart the kernel, and rerun the notebook.

In [2]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

try:
    import nashpy as nash
    NASH_AVAILABLE = True
except ImportError:
    nash = None
    NASH_AVAILABLE = False

print("nashpy available:", NASH_AVAILABLE)


nashpy available: True


In [3]:
# =========================
#  2. BLOCK – CONFIGURATION AND INPUT SELECTION
# =========================

RUN_MODE = "full"
RUN_PRIORITY = ["full", "pilot", "quick"]
TOLERANCE = 1e-9
MAX_MIXED_EQUILIBRIA_PER_SCENARIO = 100

PROJECT_ROOT = Path.cwd().resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def payoff_candidates(mode: str) -> list[Path]:
    filename = f"payoff_{mode}_sensitivity_scenarios.csv"
    return [RESULTS_DIR / filename, PROJECT_ROOT / filename]

def resolve_input(mode: str) -> tuple[str, Path]:
    modes = RUN_PRIORITY if mode == "auto" else [mode]
    if mode != "auto" and mode not in RUN_PRIORITY:
        raise ValueError("RUN_MODE must be 'auto', 'quick', 'pilot', or 'full'.")
    for candidate_mode in modes:
        path = next((item for item in payoff_candidates(candidate_mode) if item.exists()), None)
        if path is not None:
            return candidate_mode, path
    raise FileNotFoundError("No quick, pilot, or full sensitivity-scenario file was found.")

selected_mode, SCENARIO_PATH = resolve_input(RUN_MODE)
df_scenarios = pd.read_csv(SCENARIO_PATH)

print("Selected run mode:", selected_mode)
print("Scenario input:", SCENARIO_PATH)


Selected run mode: full
Scenario input: C:\work\LLM filtering\TDK\code\results\payoff_full_sensitivity_scenarios.csv


In [4]:
# =========================
#  3. BLOCK – VALIDATE SCENARIO DATA
# =========================

SCENARIO_COLUMNS = ["defender_scenario", "attack_cost_mode"]
STRATEGY_COLUMNS = ["strategy_id", "strategy_name"]
required_columns = {
    *SCENARIO_COLUMNS,
    *STRATEGY_COLUMNS,
    "defense_id",
    "success_signal",
    "attacker_payoff",
    "defender_payoff",
}
missing_columns = required_columns - set(df_scenarios.columns)
if missing_columns:
    raise ValueError(f"Missing scenario columns: {sorted(missing_columns)}")

duplicate_count = df_scenarios.duplicated(
    SCENARIO_COLUMNS + STRATEGY_COLUMNS + ["defense_id"]
).sum()
if duplicate_count:
    raise ValueError(f"Duplicate scenario game cells: {duplicate_count}")

for column in ["attacker_payoff", "defender_payoff"]:
    df_scenarios[column] = pd.to_numeric(df_scenarios[column], errors="coerce")
if df_scenarios[["attacker_payoff", "defender_payoff"]].isna().any().any():
    raise ValueError("At least one payoff value is missing or non-numeric.")

scenario_count = df_scenarios[SCENARIO_COLUMNS].drop_duplicates().shape[0]
strategy_count = df_scenarios[STRATEGY_COLUMNS].drop_duplicates().shape[0]
defense_count = df_scenarios["defense_id"].nunique()

print("Scenarios:", scenario_count)
print("Attack strategies:", strategy_count)
print("Defenses:", defense_count)
print("Success signals:", sorted(df_scenarios["success_signal"].unique()))

if selected_mode == "quick":
    print("WARNING: Quick-mode equilibria validate the pipeline only.")
if strategy_count > 1 and not NASH_AVAILABLE:
    print("WARNING: nashpy is unavailable; pure equilibria will be computed, but mixed analysis will be incomplete.")


Scenarios: 8
Attack strategies: 3
Defenses: 5
Success signals: ['judge_asr']


In [5]:
# =========================
#  4. BLOCK – GAME AND EQUILIBRIUM HELPERS
# =========================

def extract_game(group: pd.DataFrame):
    attacker = group.pivot(
        index=STRATEGY_COLUMNS, columns="defense_id", values="attacker_payoff"
    ).sort_index()
    defender = group.pivot(
        index=STRATEGY_COLUMNS, columns="defense_id", values="defender_payoff"
    ).reindex(index=attacker.index, columns=attacker.columns)
    if attacker.isna().any().any() or defender.isna().any().any():
        raise ValueError("Scenario does not contain a complete rectangular game.")
    return attacker, defender

def pure_equilibrium_cells(attacker: np.ndarray, defender: np.ndarray):
    attacker_best = attacker >= attacker.max(axis=0, keepdims=True) - TOLERANCE
    defender_best = defender >= defender.max(axis=1, keepdims=True) - TOLERANCE
    return [tuple(cell) for cell in np.argwhere(attacker_best & defender_best)]

def support_labels(probabilities: np.ndarray, labels: list[str]) -> list[str]:
    return [label for probability, label in zip(probabilities, labels) if probability > TOLERANCE]

def probability_json(probabilities: np.ndarray, labels: list[str]) -> str:
    mapping = {label: float(value) for label, value in zip(labels, probabilities) if value > TOLERANCE}
    return json.dumps(mapping, ensure_ascii=False, sort_keys=True)

def equilibrium_signature(
    attacker_probabilities: np.ndarray,
    defender_probabilities: np.ndarray,
    attack_labels: list[str],
    defense_labels: list[str],
) -> str:
    attack_support = support_labels(attacker_probabilities, attack_labels)
    defense_support = support_labels(defender_probabilities, defense_labels)
    return f"A[{','.join(attack_support)}]|D[{','.join(defense_support)}]"


In [6]:
# =========================
#  5. BLOCK – BEST RESPONSES AND PURE NASH EQUILIBRIA
# =========================

best_response_rows = []
pure_rows = []
game_cache = {}

for scenario_key, group in df_scenarios.groupby(SCENARIO_COLUMNS, sort=True):
    defender_scenario, attack_cost_mode = scenario_key
    attacker_df, defender_df = extract_game(group)
    attacker = attacker_df.to_numpy(dtype=float)
    defender = defender_df.to_numpy(dtype=float)
    attack_keys = list(attacker_df.index)
    attack_labels = [f"{key[0]}: {key[1]}" for key in attack_keys]
    defense_labels = [str(item) for item in attacker_df.columns]
    game_cache[scenario_key] = (attacker_df, defender_df, attack_labels, defense_labels)

    for defense_index, defense_label in enumerate(defense_labels):
        best_value = attacker[:, defense_index].max()
        responses = [
            attack_labels[index]
            for index in np.flatnonzero(attacker[:, defense_index] >= best_value - TOLERANCE)
        ]
        best_response_rows.append({
            "defender_scenario": defender_scenario,
            "attack_cost_mode": attack_cost_mode,
            "player": "attacker",
            "opponent_action": defense_label,
            "best_responses": json.dumps(responses, ensure_ascii=False),
            "best_payoff": float(best_value),
        })

    for attack_index, attack_label in enumerate(attack_labels):
        best_value = defender[attack_index, :].max()
        responses = [
            defense_labels[index]
            for index in np.flatnonzero(defender[attack_index, :] >= best_value - TOLERANCE)
        ]
        best_response_rows.append({
            "defender_scenario": defender_scenario,
            "attack_cost_mode": attack_cost_mode,
            "player": "defender",
            "opponent_action": attack_label,
            "best_responses": json.dumps(responses, ensure_ascii=False),
            "best_payoff": float(best_value),
        })

    for equilibrium_id, (attack_index, defense_index) in enumerate(
        pure_equilibrium_cells(attacker, defender), start=1
    ):
        pure_rows.append({
            "defender_scenario": defender_scenario,
            "attack_cost_mode": attack_cost_mode,
            "equilibrium_id": equilibrium_id,
            "attack_strategy": attack_labels[attack_index],
            "defense_strategy": defense_labels[defense_index],
            "attacker_payoff": float(attacker[attack_index, defense_index]),
            "defender_payoff": float(defender[attack_index, defense_index]),
            "support_signature": f"A[{attack_labels[attack_index]}]|D[{defense_labels[defense_index]}]",
        })

df_best_responses = pd.DataFrame(best_response_rows)
df_pure = pd.DataFrame(pure_rows)
display(df_pure)


,defender_scenario,attack_cost_mode,equilibrium_id,attack_strategy,defense_strategy,attacker_payoff,defender_payoff,support_signature
0,balanced,free_prompt_creation,1,S2: Stealthy Dependency Injection,D2,0.750000,0.114524,A[S2: Stealthy Dependency Injection]|D[D2]
1,balanced,free_prompt_creation,2,S3: Context Hijacking,D3,0.500000,0.213238,A[S3: Context Hijacking]|D[D3]
2,balanced,pay_per_execution,1,S2: Stealthy Dependency Injection,D2,0.736026,0.114524,A[S2: Stealthy Dependency Injection]|D[D2]
3,balanced,pay_per_execution,2,S3: Context Hijacking,D3,0.500000,0.213238,A[S3: Context Hijacking]|D[D3]
4,cost_heavy,free_prompt_creation,1,S2: Stealthy Dependency Injection,D0,0.857143,-0.040364,A[S2: Stealthy Dependency Injection]|D[D0]
5,cost_heavy,pay_per_execution,1,S2: Stealthy Dependency Injection,D0,0.843169,-0.040364,A[S2: Stealthy Dependency Injection]|D[D0]
6,false_positive_heavy,free_prompt_creation,1,S2: Stealthy Dependency Injection,D2,0.750000,0.085893,A[S2: Stealthy Dependency Injection]|D[D2]
7,false_positive_heavy,free_prompt_creation,2,S3: Context Hijacking,D3,0.500000,0.157391,A[S3: Context Hijacking]|D[D3]
8,false_positive_heavy,pay_per_execution,1,S2: Stealthy Dependency Injection,D2,0.736026,0.085893,A[S2: Stealthy Dependency Injection]|D[D2]
9,false_positive_heavy,pay_per_execution,2,S3: Context Hijacking,D3,0.500000,0.157391,A[S3: Context Hijacking]|D[D3]


In [7]:
# =========================
#  6. BLOCK – MIXED-STRATEGY EQUILIBRIA
# =========================

mixed_rows = []
mixed_complete = True

for scenario_key, cached in game_cache.items():
    defender_scenario, attack_cost_mode = scenario_key
    attacker_df, defender_df, attack_labels, defense_labels = cached
    attacker = attacker_df.to_numpy(dtype=float)
    defender = defender_df.to_numpy(dtype=float)

    if attacker.shape[0] < 2 or attacker.shape[1] < 2:
        continue
    if not NASH_AVAILABLE:
        mixed_complete = False
        continue

    game = nash.Game(attacker, defender)
    seen = set()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        equilibria = game.support_enumeration(tol=TOLERANCE)
        for attacker_probabilities, defender_probabilities in equilibria:
            attacker_probabilities = np.asarray(attacker_probabilities, dtype=float)
            defender_probabilities = np.asarray(defender_probabilities, dtype=float)
            if not np.isfinite(attacker_probabilities).all() or not np.isfinite(defender_probabilities).all():
                continue
            if (attacker_probabilities < -TOLERANCE).any() or (defender_probabilities < -TOLERANCE).any():
                continue

            attacker_probabilities = np.clip(attacker_probabilities, 0.0, None)
            defender_probabilities = np.clip(defender_probabilities, 0.0, None)
            attacker_probabilities /= attacker_probabilities.sum()
            defender_probabilities /= defender_probabilities.sum()

            if (attacker_probabilities > TOLERANCE).sum() == 1 and (defender_probabilities > TOLERANCE).sum() == 1:
                continue

            key = (
                tuple(np.round(attacker_probabilities, 10)),
                tuple(np.round(defender_probabilities, 10)),
            )
            if key in seen:
                continue
            seen.add(key)

            mixed_rows.append({
                "defender_scenario": defender_scenario,
                "attack_cost_mode": attack_cost_mode,
                "equilibrium_id": len(seen),
                "attacker_distribution": probability_json(attacker_probabilities, attack_labels),
                "defender_distribution": probability_json(defender_probabilities, defense_labels),
                "attacker_expected_payoff": float(attacker_probabilities @ attacker @ defender_probabilities),
                "defender_expected_payoff": float(attacker_probabilities @ defender @ defender_probabilities),
                "support_signature": equilibrium_signature(
                    attacker_probabilities, defender_probabilities, attack_labels, defense_labels
                ),
            })
            if len(seen) >= MAX_MIXED_EQUILIBRIA_PER_SCENARIO:
                break

df_mixed = pd.DataFrame(mixed_rows)
print("Mixed-equilibrium analysis complete:", mixed_complete)
display(df_mixed)


Mixed-equilibrium analysis complete: True


,defender_scenario,attack_cost_mode,equilibrium_id,attacker_distribution,defender_distribution,attacker_expected_payoff,defender_expected_payoff,support_signature
0,balanced,free_prompt_creation,1,"{""S1: Fake Directive Manipulation"": 0.05115059...","{""D2"": 0.8359966358284274, ""D3"": 0.16400336417...",0.681665,0.104307,"A[S1: Fake Directive Manipulation,S2: Stealthy..."
1,balanced,pay_per_execution,1,"{""S2: Stealthy Dependency Injection"": 0.938348...","{""D2"": 0.570444261423154, ""D3"": 0.429555738576...",0.557044,0.105076,"A[S2: Stealthy Dependency Injection,S3: Contex..."
2,false_positive_heavy,free_prompt_creation,1,"{""S1: Fake Directive Manipulation"": 0.06160711...","{""D2"": 0.8359966358284274, ""D3"": 0.16400336417...",0.681665,0.076664,"A[S1: Fake Directive Manipulation,S2: Stealthy..."
3,false_positive_heavy,pay_per_execution,1,"{""S2: Stealthy Dependency Injection"": 0.925745...","{""D2"": 0.570444261423154, ""D3"": 0.429555738576...",0.557044,0.077359,"A[S2: Stealthy Dependency Injection,S3: Contex..."


In [8]:
# =========================
#  7. BLOCK – SENSITIVITY AND STABILITY SUMMARY
# =========================

summary_rows = []
scenario_keys = df_scenarios[SCENARIO_COLUMNS].drop_duplicates().itertuples(index=False, name=None)

for defender_scenario, attack_cost_mode in scenario_keys:
    pure_group = (
        df_pure.loc[
            df_pure["defender_scenario"].eq(defender_scenario)
            & df_pure["attack_cost_mode"].eq(attack_cost_mode)
        ]
        if not df_pure.empty else pd.DataFrame()
    )
    mixed_group = (
        df_mixed.loc[
            df_mixed["defender_scenario"].eq(defender_scenario)
            & df_mixed["attack_cost_mode"].eq(attack_cost_mode)
        ]
        if not df_mixed.empty else pd.DataFrame()
    )

    pure_signatures = sorted(pure_group["support_signature"].tolist()) if not pure_group.empty else []
    mixed_signatures = sorted(mixed_group["support_signature"].tolist()) if not mixed_group.empty else []
    summary_rows.append({
        "defender_scenario": defender_scenario,
        "attack_cost_mode": attack_cost_mode,
        "pure_equilibrium_count": len(pure_signatures),
        "mixed_equilibrium_count": len(mixed_signatures),
        "pure_supports": json.dumps(pure_signatures, ensure_ascii=False),
        "mixed_supports": json.dumps(mixed_signatures, ensure_ascii=False),
        "mixed_analysis_complete": mixed_complete,
    })

df_summary = pd.DataFrame(summary_rows).sort_values(SCENARIO_COLUMNS).reset_index(drop=True)
display(df_summary)

unique_pure_support_sets = df_summary["pure_supports"].nunique()
unique_mixed_support_sets = df_summary["mixed_supports"].nunique()
equilibrium_changes = unique_pure_support_sets > 1 or unique_mixed_support_sets > 1

print("Equilibrium support changes across scenarios:", equilibrium_changes)
if equilibrium_changes:
    print("CONDITIONAL RESULT: equilibrium conclusions depend on payoff weights or attacker-cost assumptions.")
else:
    print("The observed equilibrium support is stable across the evaluated scenarios.")


,defender_scenario,attack_cost_mode,pure_equilibrium_count,mixed_equilibrium_count,pure_supports,mixed_supports,mixed_analysis_complete
0,balanced,free_prompt_creation,2,1,"[""A[S2: Stealthy Dependency Injection]|D[D2]"",...","[""A[S1: Fake Directive Manipulation,S2: Stealt...",True
1,balanced,pay_per_execution,2,1,"[""A[S2: Stealthy Dependency Injection]|D[D2]"",...","[""A[S2: Stealthy Dependency Injection,S3: Cont...",True
2,cost_heavy,free_prompt_creation,1,0,"[""A[S2: Stealthy Dependency Injection]|D[D0]""]",[],True
3,cost_heavy,pay_per_execution,1,0,"[""A[S2: Stealthy Dependency Injection]|D[D0]""]",[],True
4,false_positive_heavy,free_prompt_creation,2,1,"[""A[S2: Stealthy Dependency Injection]|D[D2]"",...","[""A[S1: Fake Directive Manipulation,S2: Stealt...",True
5,false_positive_heavy,pay_per_execution,2,1,"[""A[S2: Stealthy Dependency Injection]|D[D2]"",...","[""A[S2: Stealthy Dependency Injection,S3: Cont...",True
6,security_heavy,free_prompt_creation,1,0,"[""A[S3: Context Hijacking]|D[D3]""]",[],True
7,security_heavy,pay_per_execution,1,0,"[""A[S3: Context Hijacking]|D[D3]""]",[],True


Equilibrium support changes across scenarios: True
CONDITIONAL RESULT: equilibrium conclusions depend on payoff weights or attacker-cost assumptions.


In [ ]:
# =========================
#  8. BLOCK – SAVE GAME-THEORETIC OUTPUTS
# =========================

OUTPUT_PREFIX = RESULTS_DIR / f"game_{selected_mode}"
output_files = {
    "best_responses": Path(f"{OUTPUT_PREFIX}_best_responses.csv"),
    "pure_equilibria": Path(f"{OUTPUT_PREFIX}_pure_equilibria.csv"),
    "mixed_equilibria": Path(f"{OUTPUT_PREFIX}_mixed_equilibria.csv"),
    "sensitivity_summary": Path(f"{OUTPUT_PREFIX}_sensitivity_summary.csv"),
}

df_best_responses.to_csv(output_files["best_responses"], index=False)
df_pure.to_csv(output_files["pure_equilibria"], index=False)
df_mixed.to_csv(output_files["mixed_equilibria"], index=False)
df_summary.to_csv(output_files["sensitivity_summary"], index=False)

for name, path in output_files.items():
    print(f"{name}: {path}")


best_responses: C:\work\LLM filtering\TDK\code\results\game_full_best_responses.csv
pure_equilibria: C:\work\LLM filtering\TDK\code\results\game_full_pure_equilibria.csv
mixed_equilibria: C:\work\LLM filtering\TDK\code\results\game_full_mixed_equilibria.csv
sensitivity_summary: C:\work\LLM filtering\TDK\code\results\game_full_sensitivity_summary.csv


: 

## Interpretation

An equilibrium is a best-response pair under one explicit payoff specification. It is not automatically a universal security recommendation. If equilibrium supports change across the saved scenarios, report conclusions conditionally—for example, one defense may be rational under security-heavy assumptions and another under cost-heavy assumptions.

Final TDK claims should rely on pilot/full measurements with multiple attack strategies, sufficient attack and benign samples, and preferably end-to-end judge-based attack success. Pass-rate-based games must be described as defense-bypass games rather than full attack-success games.